In [11]:
import os
import json
import uuid
import logging
from celery import shared_task
from pystac_monty.sources.emdat import EMDATTransformer, EMDATDataSource
from pystac_monty.geocoding import GAULGeocoder

logger = logging.getLogger(__name__)
excel_file_path = "emdat_public_2022_08_11_query_uid-IKGfZc.xlsx"
data_source = EMDATDataSource(
    source_url="https://public.emdat.be",
    data=excel_file_path
)

geocoder = None 
transformer = EMDATTransformer(data_source, geocoder)
stac_items = transformer.make_items()

output_dir = "emdat_stac_items"
os.makedirs(output_dir, exist_ok=True)

for item in stac_items:
    with open(f"{output_dir}/{item.id}.json", "w") as f:
        json.dump(item.to_dict(), f, indent=2)
        
collection_and_item_type_map = {
    "emdat-events": "EVENT",  # PyStacLoadData.ItemType.EVENT,
    "emdat-hazards": "HAZARD",  # PyStacLoadData.ItemType.HAZARD,
    "emdat-impacts": "IMPACT",  # PyStacLoadData.ItemType.IMPACT,
}


@shared_task
def transform_emdat_data(excel_file_path, **kwargs):
    """
    Transform EMDAT Excel data to STAC items.
    """
    data_source = EMDATDataSource(
        source_url="https://public.emdat.be",
        data=excel_file_path
    )
    
    return transform_data(
        "EMDAT",  
        EMDATTransformer,
        lambda **kwargs: data_source, 
        "extraction_id",  
        None,  
    )


@shared_task
def transform_data(source, transformer_class, data_source_factory, extraction_id, data):
    logger.info(f"Transformation started for {source} data")
    
    transform_obj = {"id": ""}

    geocoder = None  

    try:
        data_source_instance = data_source_factory(source_url="https://public.emdat.be", data=data)
        
        # Create transformer and generate STAC items
        transformer = transformer_class(data_source_instance, geocoder)
        transformed_items = transformer.make_items()
        
        print(f"Generated {len(transformed_items)} STAC items")
        
        output_dir = "emdat_stac_items"
        os.makedirs(output_dir, exist_ok=True)
        
        transformed_item_list = []
        
        for item in transformed_items:
            item_type = collection_and_item_type_map.get(item.collection_id, "UNKNOWN")
            transformed_item_dict = item.to_dict()
            transformed_item_dict["properties"]["monty:etl_id"] = str(uuid.uuid4())
            
            with open(f"{output_dir}/{item.id}.json", "w") as f:
                json.dump(transformed_item_dict, f, indent=2)
            
            transformed_item_list.append({
                "transform_id": transform_obj["id"],
                "item": transformed_item_dict,
                "collection_id": item.collection_id,
                "item_type": item_type,
                "load_status": "PENDING"
            })
            
    except Exception as e:
        logger.error("Transformation failed", exc_info=True, extra={"source": source})
        raise e

    logger.info(f"Transformation ended for {source} data")
    print(f"Successfully generated {len(transformed_items)} STAC items in {output_dir}")
    return transformed_item_list


if __name__ == "__main__":
    transform_emdat_data("emdat_public_2022_08_11_query_uid-IKGfZc.xlsx")
print(f"Successfully generated {len(stac_items)} STAC items in {output_dir}")

/home/runner/workspace/.pythonlibs/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/runner/workspace/.pythonlibs/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
INFO:__main__:Transformation started for EMDAT data
INFO:__main__:Transformation ended for EMDAT data


Generated 0 STAC items
Successfully generated 0 STAC items in emdat_stac_items
Successfully generated 0 STAC items in emdat_stac_items
